# El sábado que contrató una cuadrilla

**Nivel:** intermediate

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/narrative/blob/codex/corporate-data-narrative-lab/corporate-data-narrative-lab/outputs/notebooks/13-el-sabado-que-contrato-una-cuadrilla.ipynb)

## Pregunta central

¿Basta un sábado récord para asignar de forma permanente una cuadrilla adicional a los días no laborales?

## Recreación narrativa

> **Marta:** "El sábado rompimos récord. Ya pedí una cuadrilla extra para todos los fines."
>
> **Elisa:** "¿Cuántos sábados miraste?"
>
> **Bruno:** "El de la presentación. El contrato vence hoy."
>
> **Elisa:** "Entonces el dato tiene fecha de entrega, no tamaño de muestra."

*La escena es una recreación; las conclusiones provienen del dataset citado.*

## Fuente real

**Bike Sharing Dataset — day.csv**, UCI Machine Learning Repository; Hadi Fanaee-T. [Página de origen](https://archive.ics.uci.edu/dataset/275/bike%2Bsharing%2Bdataset) · [datos](https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip) · licencia: CC BY 4.0.  
Consultado: 2026-07-18 · 731 filas · columnas usadas: `instant`, `dteday`, `season`, `yr`, `mnth`, `holiday`, `weekday`, `workingday`, `weathersit`, `temp`, `atemp`, `hum`, `windspeed`, `casual`, `registered`, `cnt`.

In [1]:
# @title Preparar los datos { display-mode: "form" }
periodo = "Ambos años" # @param ["Ambos años", "2011", "2012"]
umbral = 5000 # @param {type:"slider", min:3000, max:7000, step:250}
dias_ventana = 30 # @param {type:"slider", min:10, max:60, step:5}
tamano_muestra = 60 # @param {type:"slider", min:20, max:120, step:10}
confianza = 95 # @param [90, 95, 99]
alpha = 0.05 # @param [0.10, 0.05, 0.01]
delta_relevante = 500 # @param {type:"slider", min:100, max:1000, step:50}
repeticiones = 2000 # @param [1000, 2000, 5000]
import io, zipfile, hashlib
from math import comb, factorial, erf, sqrt, exp, pi
from urllib.request import urlopen
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
DATA_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
DATA_MEMBER = "day.csv"
DATA_SHA256 = "b70182d0d0508e9abbb79306ce5c0cec34869000f8220175ac83d11dbe845401"
archive_bytes = urlopen(DATA_URL).read()
if hashlib.sha256(archive_bytes).hexdigest() != DATA_SHA256:
    raise ValueError("El archivo de UCI cambió; revisa la fuente y la huella antes de continuar.")
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    df_completo = pd.read_csv(archive.open(DATA_MEMBER), parse_dates=["dteday"])
year_code = {"2011": 0, "2012": 1}.get(periodo)
df = df_completo if year_code is None else df_completo.query("yr == @year_code")
df = df.reset_index(drop=True).copy()
df["tipo_dia"] = np.where(df["workingday"].eq(1), "Laboral", "No laboral")
df["alta_demanda"] = df["cnt"].ge(umbral)
AZUL, NARANJA, TINTA, NEUTRO = "#2F6B7C", "#D97735", "#173F5F", "#A7B4BC"
def normal_pdf(x, mean, sd):
    return np.exp(-0.5 * ((x - mean) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
def normal_cdf(value):
    return (1 + erf(value / sqrt(2))) / 2
def binomial_values(n, p):
    k = np.arange(n + 1)
    return k, np.array([comb(n, int(i)) * p**i * (1-p)**(n-i) for i in k])
def poisson_values(rate, maximum):
    k = np.arange(maximum + 1)
    return k, np.array([exp(-rate) * rate**int(i) / factorial(int(i)) for i in k])
def repeated_sample_means(series, n, reps, seed):
    values, rng = series.dropna().to_numpy(), np.random.default_rng(seed)
    return np.array([rng.choice(values, n, replace=False).mean() for _ in range(reps)])
def bootstrap_difference(first, second, reps, seed):
    rng = np.random.default_rng(seed)
    return np.array([rng.choice(first, len(first), True).mean() - rng.choice(second, len(second), True).mean() for _ in range(reps)])
def permutation_differences(values, first_size, reps, seed):
    rng, results = np.random.default_rng(seed), []
    for _ in range(reps):
        shuffled = rng.permutation(values)
        results.append(shuffled[:first_size].mean() - shuffled[first_size:].mean())
    return np.asarray(results)
def approximate_power(delta, standard_error, significance):
    critical = {0.10: 1.645, 0.05: 1.96, 0.01: 2.576}[round(significance, 2)]
    shift = abs(delta) / standard_error
    return normal_cdf(-critical - shift) + 1 - normal_cdf(critical - shift)
display(Markdown(f"**Vista:** {periodo} · **{len(df)} días** · evento = **{umbral:,}+ viajes** · muestra = **{tamano_muestra} días**"))
df[["dteday", "tipo_dia", "weathersit", "cnt", "alta_demanda"]].head()

**Vista:** Ambos años · **731 días** · evento = **5,000+ viajes** · muestra = **60 días**

,dteday,tipo_dia,weathersit,cnt,alta_demanda
0,2011-01-01,No laboral,2,985,False
1,2011-01-02,No laboral,2,801,False
2,2011-01-03,Laboral,1,1349,False
3,2011-01-04,Laboral,1,1562,False
4,2011-01-05,Laboral,1,1600,False


## 1. Probabilidad básica

> **Bruno:** "El sábado récord ya trae uniforme y horario."
>
> **Elisa:** "Primero necesita denominador."
>
> **Marta:** "¿El denominador firma hoy?"


**Pregunta:** ¿Qué tan probable es superar el umbral y cambia esa probabilidad entre días laborales y no laborales?

**Conexión:** Punto de partida: convertir un sábado récord en un evento medible antes de firmar un turno permanente.

In [2]:
evento = df["cnt"].ge(umbral)
p_evento = evento.mean()
p_complemento = 1 - p_evento
p_laboral = df["workingday"].eq(1).mean()
p_conjunta = (evento & df["workingday"].eq(1)).mean()
p_condicional = df.groupby("tipo_dia")["alta_demanda"].mean()
independencia = p_conjunta - p_evento * p_laboral
probabilidades = pd.Series({"P(evento)": p_evento, "P(complemento)": p_complemento, "P(evento∩laboral)-P(evento)P(laboral)": independencia, **{f"P(evento|{k})": v for k, v in p_condicional.items()}}); probabilidades.to_frame("probabilidad")

,probabilidad
P(evento),0.391245
P(complemento),0.608755
P(evento∩laboral)-P(evento)P(laboral),-0.000851
P(evento|Laboral),0.390000
P(evento|No laboral),0.393939


In [3]:
# @title Explorar Probabilidad básica { display-mode: "form" }
orden = ["Laboral", "No laboral"]
evento_pct = np.array([p_evento, p_complemento]); evento_n = np.array([evento.sum(), (~evento).sum()])
cond_pct = p_condicional.reindex(orden).to_numpy(); cond_n = df.groupby("tipo_dia")["alta_demanda"].sum().reindex(orden).to_numpy()
tamanos = df["tipo_dia"].value_counts().reindex(orden).to_numpy(); esperado_n = tamanos * p_evento
fig = make_subplots(rows=1, cols=2, subplot_titles=("Evento y complemento", "Laboral frente a no laboral"))
fig.add_trace(go.Bar(x=[f"{umbral:,}+ viajes", "Menos del umbral"], y=evento_pct, text=[f"{v:.1%}" for v in evento_pct], textposition="auto", marker={"color": [AZUL, "#D9E3E7"], "line": {"color": TINTA, "width": 1}}, showlegend=False, hovertemplate="%{x}<br>probabilidad=%{y:.1%}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=orden, y=cond_pct, text=[f"{v:.1%}" for v in cond_pct], textposition="auto", marker={"color": [AZUL, NARANJA], "line": {"color": TINTA, "width": 1}}, showlegend=False, hovertemplate="%{x}<br>P(evento|tipo)=%{y:.1%}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=orden, y=[p_evento, p_evento], mode="lines+markers", name="Referencia de independencia", line={"color": TINTA, "dash": "dot", "width": 2}, marker={"symbol": "diamond", "size": 8}, hovertemplate="P(evento) marginal=%{y:.1%}<extra></extra>"), row=1, col=2)
pct_title = f"Probabilidad de un día con {umbral:,}+ viajes<br><sup>{periodo} · n={len(df)} días · independencia exige P(evento|tipo)=P(evento)</sup>"
count_title = f"Días observados frente a los esperados por independencia<br><sup>{periodo} · n={len(df)} días · umbral={umbral:,} viajes</sup>"
prob_hover = ["%{x}<br>probabilidad=%{y:.1%}<extra></extra>", "%{x}<br>P(evento|tipo)=%{y:.1%}<extra></extra>", "P(evento) marginal=%{y:.1%}<extra></extra>"]
count_hover = ["%{x}<br>días=%{y:,.0f}<extra></extra>", "%{x}<br>días con evento=%{y:,.0f}<extra></extra>", "esperados por independencia=%{y:,.1f}<extra></extra>"]
buttons = [{"label": "Probabilidad", "method": "update", "args": [{"y": [evento_pct, cond_pct, [p_evento, p_evento]], "text": [[f"{v:.1%}" for v in evento_pct], [f"{v:.1%}" for v in cond_pct], None], "hovertemplate": prob_hover}, {"title.text": pct_title, "yaxis.title.text": "Probabilidad", "yaxis2.title.text": "Probabilidad", "yaxis.tickformat": ".0%", "yaxis2.tickformat": ".0%", "yaxis.range": [0, 1], "yaxis2.range": [0, 1]}]}, {"label": "Conteo de días", "method": "update", "args": [{"y": [evento_n, cond_n, esperado_n], "text": [[f"{int(v)}" for v in evento_n], [f"{int(v)}" for v in cond_n], None], "hovertemplate": count_hover}, {"title.text": count_title, "yaxis.title.text": "Días", "yaxis2.title.text": "Días", "yaxis.tickformat": ",.0f", "yaxis2.tickformat": ",.0f", "yaxis.range": [0, max(evento_n) * 1.2], "yaxis2.range": [0, max(tamanos) * 1.05]}]}]
fig.update_layout(template="plotly_white", height=500, title=pct_title, showlegend=True, legend={"orientation": "h", "y": -0.22}, margin={"t": 115, "b": 115}, font={"family": "Arial", "color": TINTA}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.21, "xanchor": "right", "buttons": buttons}])
fig.update_xaxes(title_text="Resultado del evento", row=1, col=1); fig.update_xaxes(title_text="Tipo de día", row=1, col=2)
fig.update_yaxes(title_text="Probabilidad", range=[0, 1], tickformat=".0%", row=1, col=1); fig.update_yaxes(title_text="Probabilidad", range=[0, 1], tickformat=".0%", row=1, col=2)
fig.show()


display(Markdown("**Lo que muestra:** Con ambos años y 5,000 viajes, el evento ocurre en 39.1% de los días y su complemento en 60.9%. La probabilidad condicional es 39.0% en días laborales y 39.4% en no laborales: para este umbral, el récord sabatino no implica una frecuencia sabatina mayor."))

**Lo que muestra:** Con ambos años y 5,000 viajes, el evento ocurre en 39.1% de los días y su complemento en 60.9%. La probabilidad condicional es 39.0% en días laborales y 39.4% en no laborales: para este umbral, el récord sabatino no implica una frecuencia sabatina mayor.

## 2. Variables aleatorias

> **Bruno:** "Ponle normal; se oye estable."
>
> **Elisa:** "Normal no significa bien portada."
>
> **Marta:** "Entonces usamos la que responda la pregunta, no la que cabe en la presentación."


**Pregunta:** ¿Qué variable responde cada modelo Bernoulli, binomial, normal y Poisson?

**Conexión:** Probabilidad básica produjo p(evento); ahora la usamos para describir un día, una ventana de días y una media muestral sin confundir modelos con hechos.

In [4]:
p = df["alta_demanda"].mean()
n = dias_ventana
media, desviacion = df["cnt"].mean(), df["cnt"].std(ddof=1)
parametros = pd.DataFrame({
    "variable": ["Bernoulli: un día", "Binomial: días con alta demanda", "Normal: media de n días", "Poisson: aproximación al conteo"],
    "parámetros": [f"p={p:.3f}", f"n={n}, p={p:.3f}, E={n*p:.2f}", f"μ={media:.0f}, SE={desviacion/np.sqrt(n):.1f}", f"λ={n*p:.2f}"],
})
parametros

,variable,parámetros
0,Bernoulli: un día,p=0.391
1,Binomial: días con alta demanda,"n=30, p=0.391, E=11.74"
2,Normal: media de n días,"μ=4504, SE=353.7"
3,Poisson: aproximación al conteo,λ=11.74


In [5]:
# @title Explorar Variables aleatorias { display-mode: "form" }
assert len(parametros) == 4
k_bin, pmf_bin = binomial_values(n, p); k_poi, pmf_poi = poisson_values(n*p, n)
x_normal = np.linspace(media - 4*desviacion/np.sqrt(n), media + 4*desviacion/np.sqrt(n), 250)
y_normal = normal_pdf(x_normal, media, desviacion/np.sqrt(n))
fig = make_subplots(rows=2, cols=2, subplot_titles=("Bernoulli: un día", "Binomial: conteo exacto", "Normal: media muestral", "Poisson: aproximación"), vertical_spacing=.22)
fig.add_trace(go.Bar(x=[0, 1], y=[1-p, p], marker={"color": ["#D9E3E7", AZUL], "line": {"color": TINTA, "width": 1}}, text=[f"{1-p:.1%}", f"{p:.1%}"], textposition="auto", name="Bernoulli", hovertemplate="resultado=%{x}<br>P=%{y:.1%}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=k_bin, y=pmf_bin, marker={"color": AZUL, "line": {"color": TINTA, "width": .5}}, name="Binomial", hovertemplate="días=%{x}<br>P=%{y:.2%}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=x_normal, y=y_normal, mode="lines", fill="tozeroy", line={"color": NARANJA, "width": 3}, fillcolor="rgba(217,119,53,.18)", name="Normal", hovertemplate="media=%{x:,.0f}<br>densidad=%{y:.5f}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Bar(x=k_poi, y=pmf_poi, marker={"color": NARANJA, "line": {"color": TINTA, "width": .5}}, opacity=.65, name="Poisson", hovertemplate="días=%{x}<br>P=%{y:.2%}<extra></extra>"), row=2, col=2)
fig.add_trace(go.Scatter(x=k_bin, y=pmf_bin, mode="lines", line={"color": TINTA, "dash": "dot", "width": 2}, name="Binomial de referencia", hovertemplate="Binomial<br>días=%{x}<br>P=%{y:.2%}<extra></extra>"), row=2, col=2)
title = f"Cuatro variables aleatorias para la misma decisión<br><sup>{periodo} · evento={umbral:,}+ viajes · ventana={n} días · los modelos describen variables distintas</sup>"
buttons = [{"label": "Vista completa", "method": "update", "args": [{"opacity": [1, 1, 1, .65, 1]}, {"title.text": title}]}, {"label": "Resaltar exactas", "method": "update", "args": [{"opacity": [1, 1, .2, .2, .2]}, {"title.text": title.replace("Cuatro variables", "Bernoulli y binomial: variables exactas")}]}, {"label": "Resaltar aproximaciones", "method": "update", "args": [{"opacity": [.2, .2, 1, .8, 1]}, {"title.text": title.replace("Cuatro variables", "Normal y Poisson: aproximaciones") }]}]
fig.update_layout(template="plotly_white", height=650, title=title, legend={"orientation": "h", "y": -0.14}, margin={"t": 120, "b": 105}, font={"family": "Arial", "color": TINTA}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.14, "xanchor": "right", "buttons": buttons}])
fig.update_xaxes(title_text="Resultado del evento", tickmode="array", tickvals=[0, 1], ticktext=["0 · no", "1 · sí"], range=[-.5, 1.5], row=1, col=1); fig.update_yaxes(title_text="Probabilidad", tickformat=".0%", row=1, col=1)
fig.update_xaxes(title_text="Días con alta demanda", row=1, col=2); fig.update_yaxes(title_text="Probabilidad", tickformat=".0%", row=1, col=2)
fig.update_xaxes(title_text="Media de viajes por día", row=2, col=1); fig.update_yaxes(title_text="Densidad", row=2, col=1)
fig.update_xaxes(title_text="Días con alta demanda", row=2, col=2); fig.update_yaxes(title_text="Probabilidad", tickformat=".0%", row=2, col=2)
fig.show()


display(Markdown("**Lo que muestra:** Bernoulli representa si un día supera el umbral; binomial cuenta cuántos lo superan en 30 días; la normal aproxima la media de 30 días; Poisson aproxima un conteo con λ=11.7. Como p≈0.39 no es raro, la línea binomial recuerda que Poisson aquí es una aproximación, no una licencia para ignorar el mecanismo."))

**Lo que muestra:** Bernoulli representa si un día supera el umbral; binomial cuenta cuántos lo superan en 30 días; la normal aproxima la media de 30 días; Poisson aproxima un conteo con λ=11.7. Como p≈0.39 no es raro, la línea binomial recuerda que Poisson aquí es una aproximación, no una licencia para ignorar el mecanismo.

## 3. Muestreo

> **Bruno:** "Tomé sólo días despejados para que el clima no estorbara."
>
> **Elisa:** "Quitaste los días que sí estorban a Operaciones."
>
> **Marta:** "La muestra quedó muy limpia."
>
> **Elisa:** "Limpia y sesgada."


**Pregunta:** ¿Una muestra aleatoria converge y una muestra de días despejados responde la misma pregunta?

**Conexión:** Variables aleatorias usó parámetros estimados con los datos; ahora comprobamos cuánto varían al tomar muestras y qué ocurre si sólo elegimos días convenientes.

In [6]:
medias_aleatorias = repeated_sample_means(df["cnt"], tamano_muestra, 600, 41)
medias_despejadas = repeated_sample_means(df.query("weathersit == 1")["cnt"], tamano_muestra, 600, 42)
orden = np.random.default_rng(43).permutation(df["cnt"].to_numpy())
media_acumulada = np.cumsum(orden) / np.arange(1, len(orden) + 1)
mu = df["cnt"].mean()
mu_despejado = df.query("weathersit == 1")["cnt"].mean()
muestreo = pd.Series({"media poblacional": mu, "DE de medias aleatorias": medias_aleatorias.std(ddof=1), "media sólo despejado": mu_despejado, "sesgo de selección": mu_despejado-mu, "media acumulada final": media_acumulada[-1]})
muestreo.to_frame("viajes")

,viajes
media poblacional,4504.348837
DE de medias aleatorias,234.460561
media sólo despejado,4876.786177
sesgo de selección,372.437340
media acumulada final,4504.348837


In [7]:
# @title Explorar Muestreo { display-mode: "form" }
assert muestreo.notna().all()
fig = make_subplots(rows=1, cols=3, subplot_titles=("Variabilidad muestral", "Sesgo de selección", "Ley de grandes números"), column_widths=[.36, .25, .39], horizontal_spacing=.1)
fig.add_trace(go.Histogram(x=medias_aleatorias, nbinsx=24, histnorm="probability density", marker={"color": AZUL, "line": {"color": TINTA, "width": .5}}, opacity=.75, name="Muestras aleatorias", hovertemplate="media=%{x:,.0f}<br>densidad=%{y:.4f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Histogram(x=medias_despejadas, nbinsx=24, histnorm="probability density", marker={"color": NARANJA, "line": {"color": TINTA, "width": .5}}, opacity=.75, visible=False, name="Sólo despejado", hovertemplate="media=%{x:,.0f}<br>densidad=%{y:.4f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=["Todos", "Despejado"], y=[mu, mu_despejado], marker={"color": [AZUL, NARANJA], "line": {"color": TINTA, "width": 1}}, text=[f"{mu:,.0f}", f"{mu_despejado:,.0f}"], textposition="auto", showlegend=False, hovertemplate="%{x}<br>media=%{y:,.0f} viajes<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=np.arange(1, len(media_acumulada)+1), y=media_acumulada, mode="lines", line={"color": AZUL, "width": 3}, name="Media acumulada", hovertemplate="n=%{x}<br>media=%{y:,.0f}<extra></extra>"), row=1, col=3)
fig.add_trace(go.Scatter(x=[1, len(media_acumulada)], y=[mu, mu], mode="lines", line={"color": TINTA, "dash": "dot", "width": 2}, name="Media poblacional", hovertemplate="media poblacional=%{y:,.0f}<extra></extra>"), row=1, col=3)
title = f"Qué cambia al elegir una muestra<br><sup>{periodo} · n muestral={tamano_muestra} días · 600 repeticiones · semilla fija</sup>"
buttons = [{"label": "Muestra aleatoria", "method": "update", "args": [{"visible": [True, False, True, True, True]}, {"title.text": title}]}, {"label": "Sólo clima despejado", "method": "update", "args": [{"visible": [False, True, True, True, True]}, {"title.text": title.replace("Qué cambia", "Sesgo que aparece") }]}]
fig.update_layout(template="plotly_white", height=510, title=title, legend={"orientation": "h", "y": -0.24}, margin={"t": 115, "b": 125}, font={"family": "Arial", "color": TINTA}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.2, "xanchor": "right", "buttons": buttons}])
fig.update_xaxes(title_text="Media de la muestra", row=1, col=1); fig.update_yaxes(title_text="Densidad", row=1, col=1)
fig.update_xaxes(title_text="Población observada", tickangle=-20, row=1, col=2); fig.update_yaxes(title_text="Viajes medios", rangemode="tozero", row=1, col=2)
fig.update_xaxes(title_text="Días acumulados", row=1, col=3); fig.update_yaxes(title_text="Media acumulada", row=1, col=3)
fig.show()


display(Markdown("**Lo que muestra:** Las muestras aleatorias de 60 días fluctúan alrededor de 4,504 viajes. Elegir sólo clima despejado cambia la población observada y eleva la media a 4,877: un sesgo de +372 viajes. La ley de grandes números estabiliza una muestra aleatoria; no corrige una regla de selección sesgada."))

**Lo que muestra:** Las muestras aleatorias de 60 días fluctúan alrededor de 4,504 viajes. Elegir sólo clima despejado cambia la población observada y eleva la media a 4,877: un sesgo de +372 viajes. La ley de grandes números estabiliza una muestra aleatoria; no corrige una regla de selección sesgada.

## 4. Incertidumbre

> **Marta:** "Dame un número."
>
> **Elisa:** "Menos 255 viajes en no laborales."
>
> **Bruno:** "Perfecto."
>
> **Elisa:** "Con un intervalo que cruza cero."
>
> **Marta:** "El cero siempre llega tarde a las juntas."


**Pregunta:** ¿Qué tan precisa es la diferencia media entre días no laborales y laborales?

**Conexión:** Muestreo mostró que una estimación cambia aun sin sesgo; ahora cuantificamos esa variabilidad con error estándar, intervalo analítico y bootstrap.

In [8]:
no_laboral, laboral = df.query("workingday == 0")["cnt"].to_numpy(), df.query("workingday == 1")["cnt"].to_numpy()
diferencia = no_laboral.mean() - laboral.mean()
error_estandar = np.sqrt(no_laboral.var(ddof=1)/len(no_laboral) + laboral.var(ddof=1)/len(laboral))
z = {90: 1.645, 95: 1.96, 99: 2.576}[confianza]
ic_analitico = (diferencia-z*error_estandar, diferencia+z*error_estandar)
bootstrap = bootstrap_difference(no_laboral, laboral, repeticiones, 44)
ic_bootstrap = np.percentile(bootstrap, [(100-confianza)/2, 100-(100-confianza)/2])
incertidumbre = pd.Series({"diferencia no laboral - laboral": diferencia, "error estándar": error_estandar, f"IC analítico {confianza}% inferior": ic_analitico[0], f"IC analítico {confianza}% superior": ic_analitico[1], f"bootstrap {confianza}% inferior": ic_bootstrap[0], f"bootstrap {confianza}% superior": ic_bootstrap[1]}); incertidumbre.to_frame("viajes/día")

,viajes/día
diferencia no laboral - laboral,-254.651169
error estándar,159.020614
IC analítico 95% inferior,-566.331571
IC analítico 95% superior,57.029234
bootstrap 95% inferior,-584.914560
bootstrap 95% superior,60.201283


In [9]:
# @title Explorar Incertidumbre { display-mode: "form" }
assert incertidumbre.notna().all()
counts, edges = np.histogram(bootstrap, bins=34, density=True); ymax = counts.max() * 1.08
fig = make_subplots(rows=1, cols=2, subplot_titles=("Distribución bootstrap", "Dos intervalos para la diferencia"), column_widths=[.62, .38], horizontal_spacing=.13)
fig.add_trace(go.Histogram(x=bootstrap, nbinsx=34, histnorm="probability density", marker={"color": AZUL, "line": {"color": TINTA, "width": .5}}, opacity=.72, name="Bootstrap", hovertemplate="diferencia=%{x:,.0f}<br>densidad=%{y:.5f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=[diferencia, diferencia], y=[0, ymax], mode="lines", line={"color": NARANJA, "width": 3}, name="Diferencia observada", hovertemplate=f"observada={diferencia:,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=list(ic_analitico), y=["Analítico", "Analítico"], mode="lines+markers", line={"color": TINTA, "width": 5}, marker={"size": 10, "symbol": "diamond"}, name="IC analítico", hovertemplate="límite=%{x:,.0f}<extra>analítico</extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=list(ic_bootstrap), y=["Bootstrap", "Bootstrap"], mode="lines+markers", line={"color": AZUL, "width": 5}, marker={"size": 10, "symbol": "circle-open"}, name="IC bootstrap", hovertemplate="límite=%{x:,.0f}<extra>bootstrap</extra>"), row=1, col=2)
fig.add_vline(x=0, line={"color": NEUTRO, "dash": "dot", "width": 2}, row=1, col=2)
title = f"Incertidumbre de la diferencia media<br><sup>{periodo} · no laboral − laboral · confianza={confianza}% · bootstrap={repeticiones:,} réplicas</sup>"
buttons = [{"label": "Comparar ambos", "method": "update", "args": [{"opacity": [.72, 1, 1, 1]}, {"title.text": title}]}, {"label": "Destacar analítico", "method": "update", "args": [{"opacity": [.2, .35, 1, .2]}, {"title.text": title.replace("Incertidumbre", "Intervalo analítico")}]}, {"label": "Destacar bootstrap", "method": "update", "args": [{"opacity": [.82, 1, .2, 1]}, {"title.text": title.replace("Incertidumbre", "Intervalo bootstrap")}]}]
fig.update_layout(template="plotly_white", height=510, title=title, legend={"orientation": "h", "y": -0.24}, margin={"t": 115, "b": 125}, font={"family": "Arial", "color": TINTA}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.2, "xanchor": "right", "buttons": buttons}])
fig.update_xaxes(title_text="Diferencia de viajes por día", row=1, col=1); fig.update_yaxes(title_text="Densidad", row=1, col=1)
fig.update_xaxes(title_text="No laboral − laboral (viajes/día)", row=1, col=2); fig.update_yaxes(title_text="Método", row=1, col=2)
fig.show()


display(Markdown("**Lo que muestra:** La diferencia observada es −255 viajes por día y su error estándar es 159. Con 95% de confianza, el intervalo analítico va aproximadamente de −566 a 57; el bootstrap cuenta la misma historia y también cruza cero. El signo observado no es una frontera nítida para contratar."))

**Lo que muestra:** La diferencia observada es −255 viajes por día y su error estándar es 159. Con 95% de confianza, el intervalo analítico va aproximadamente de −566 a 57; el bootstrap cuenta la misma historia y también cruza cero. El signo observado no es una frontera nítida para contratar.

## 5. Pruebas de hipótesis

> **Marta:** "¿El p-value es menor a .05?"
>
> **Elisa:** "No."
>
> **Bruno:** "Entonces son iguales."
>
> **Elisa:** "Tampoco. La prueba no separó esta diferencia del ruido."
>
> **Marta:** "Qué poco cooperativo."


**Pregunta:** ¿La diferencia observada contradice H0 y qué cambio relevante podía detectar el diseño?

**Conexión:** Incertidumbre dejó al cero dentro del intervalo; ahora fijamos H0, H1 y α, calculamos el p-value y distinguimos error tipo I, error tipo II y potencia.

In [10]:
hipotesis = {"H0": "μ no laboral = μ laboral", "H1": "las medias son distintas"}
permutaciones = permutation_differences(df["cnt"].to_numpy(), len(no_laboral), repeticiones, 45)
p_value = (np.count_nonzero(np.abs(permutaciones) >= abs(diferencia)) + 1) / (len(permutaciones) + 1)
pooled_sd = np.sqrt(((len(no_laboral)-1)*no_laboral.var(ddof=1) + (len(laboral)-1)*laboral.var(ddof=1)) / (len(df)-2))
cohen_d = diferencia / pooled_sd
potencia = approximate_power(delta_relevante, error_estandar, alpha)
prueba = pd.Series({**hipotesis, "diferencia observada": diferencia, "p-value permutación": p_value, "Cohen d": cohen_d, "α": alpha, "diferencia relevante": delta_relevante, "potencia aproximada": potencia})
prueba.to_frame("resultado")

,resultado
H0,μ no laboral = μ laboral
H1,las medias son distintas
diferencia observada,-254.651169
p-value permutación,0.102449
Cohen d,-0.131609
α,0.05
diferencia relevante,500
potencia aproximada,0.881842


In [11]:
# @title Explorar Pruebas de hipótesis { display-mode: "form" }
assert prueba.notna().all()
critico_z = {0.10: 1.645, 0.05: 1.96, 0.01: 2.576}[round(alpha, 2)]; critico = critico_z * error_estandar
x_test = np.linspace(-4*error_estandar, delta_relevante + 4*error_estandar, 450)
h0 = normal_pdf(x_test, 0, error_estandar); h1 = normal_pdf(x_test, delta_relevante, error_estandar)
type_ii = np.where(np.abs(x_test) < critico, h1, np.nan)
hist_counts, hist_edges = np.histogram(permutaciones, bins=34); bin_widths = np.diff(hist_edges)
hist_centers = (hist_edges[:-1] + hist_edges[1:]) / 2; hist_density = hist_counts / (len(permutaciones) * bin_widths)
extremos = permutaciones[np.abs(permutaciones) >= abs(diferencia)]
tail_counts, _ = np.histogram(extremos, bins=hist_edges); tail_density = tail_counts / (len(permutaciones) * bin_widths)
perm_ymax = hist_density.max() * 1.08
fig = make_subplots(rows=1, cols=2, subplot_titles=("Distribución nula y p-value", "Errores y potencia del diseño"), horizontal_spacing=.12)
fig.add_trace(go.Bar(x=hist_centers, y=hist_density, width=bin_widths, marker={"color": NEUTRO, "line": {"color": TINTA, "width": .4}}, opacity=.65, name="H0 por permutación", hovertemplate="diferencia bajo H0=%{x:,.0f}<br>densidad=%{y:.5f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=hist_centers, y=tail_density, width=bin_widths, marker={"color": NARANJA, "line": {"color": TINTA, "width": .4}}, opacity=.85, name="Colas del p-value", hovertemplate="cola bilateral<br>%{x:,.0f}<br>densidad=%{y:.5f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=[diferencia, diferencia], y=[0, perm_ymax], mode="lines", line={"color": TINTA, "width": 3}, name="Observado", hovertemplate=f"observado={diferencia:,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=x_test, y=h0, mode="lines", line={"color": TINTA, "width": 3}, name="H0", hovertemplate="H0<br>diferencia=%{x:,.0f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=x_test, y=h1, mode="lines", line={"color": AZUL, "width": 3}, name=f"H1: Δ={delta_relevante:,}", hovertemplate="H1<br>diferencia=%{x:,.0f}<extra></extra>"), row=1, col=2)
left_i, right_i = x_test <= -critico, x_test >= critico
fig.add_trace(go.Scatter(x=x_test[left_i], y=h0[left_i], mode="lines", fill="tozeroy", line={"color": NARANJA, "width": 1}, fillcolor="rgba(217,119,53,.26)", name="Error tipo I (α)", legendgroup="tipo_i", hovertemplate="rechazo bajo H0<extra>tipo I</extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=x_test[right_i], y=h0[right_i], mode="lines", fill="tozeroy", line={"color": NARANJA, "width": 1}, fillcolor="rgba(217,119,53,.26)", name="Error tipo I (α)", legendgroup="tipo_i", showlegend=False, hovertemplate="rechazo bajo H0<extra>tipo I</extra>"), row=1, col=2)
fig.add_trace(go.Scatter(x=x_test, y=type_ii, mode="lines", fill="tozeroy", line={"color": AZUL, "width": 1}, fillcolor="rgba(47,107,124,.18)", name="Error tipo II (β)", hovertemplate="no rechazo bajo H1<extra>tipo II</extra>"), row=1, col=2)
fig.add_vline(x=-critico, line={"color": NEUTRO, "dash": "dot"}, row=1, col=2); fig.add_vline(x=critico, line={"color": NEUTRO, "dash": "dot"}, row=1, col=2)
fig.add_annotation(x=delta_relevante, y=max(h1)*.88, text=f"potencia≈{potencia:.0%}", showarrow=False, font={"color": AZUL}, row=1, col=2)
title = f"Prueba predefinida y capacidad de detección<br><sup>{periodo} · H0: diferencia=0 · p={p_value:.3f} · d={cohen_d:.2f} · α={alpha:.2f} · potencia para Δ={delta_relevante:,}: {potencia:.0%}</sup>"
buttons = [{"label": "Ambos", "method": "update", "args": [{"opacity": [.65, .85, 1, 1, 1, 1, 1, 1]}, {"title.text": title}]}, {"label": "Resultado observado", "method": "update", "args": [{"opacity": [.8, .95, 1, .18, .18, .18, .18, .18]}, {"title.text": title.replace("Prueba predefinida", "p-value por permutación")}]}, {"label": "Errores y potencia", "method": "update", "args": [{"opacity": [.16, .16, .2, 1, 1, 1, 1, 1]}, {"title.text": title.replace("Prueba predefinida", "Diseño con errores I y II") }]}]
fig.update_layout(template="plotly_white", barmode="overlay", height=530, title=title, legend={"orientation": "h", "y": -0.27}, margin={"t": 120, "b": 145}, font={"family": "Arial", "color": TINTA}, hoverlabel={"bgcolor": "#FFFFFF"}, updatemenus=[{"x": 1, "y": 1.2, "xanchor": "right", "buttons": buttons}])
fig.update_xaxes(title_text="No laboral − laboral", row=1, col=1); fig.update_yaxes(title_text="Densidad", row=1, col=1)
fig.update_xaxes(title_text="Diferencia estimada (viajes/día)", row=1, col=2); fig.update_yaxes(title_text="Densidad aproximada", row=1, col=2)
fig.show()


display(Markdown("**Lo que muestra:** La prueba bilateral por permutación produce p≈0.10 y el efecto es pequeño (d≈−0.13): no rechazamos H0, pero el p-value no es la probabilidad de que H0 sea cierta. Tipo I sería contratar sin diferencia real; tipo II, ignorar una diferencia relevante. Para Δ=500 y α=.05, la potencia aproximada es 88%; es una capacidad planificada, no 'potencia observada'."))

**Lo que muestra:** La prueba bilateral por permutación produce p≈0.10 y el efecto es pequeño (d≈−0.13): no rechazamos H0, pero el p-value no es la probabilidad de que H0 sea cierta. Tipo I sería contratar sin diferencia real; tipo II, ignorar una diferencia relevante. Para Δ=500 y α=.05, la potencia aproximada es 88%; es una capacidad planificada, no 'potencia observada'.

## Cómo se conecta todo

La probabilidad convierte el récord en una frecuencia; las variables aleatorias describen resultados posibles; el muestreo muestra cuánto cambia una estimación y cómo se sesga; los intervalos cuantifican precisión; y la prueba obliga a separar señal, error y capacidad de detección.

## Decisión

No crear un turno permanente para días no laborales con esta comparación agregada. Mantener cobertura flexible y diseñar un piloto por estación y temporada con duración, diferencia relevante y regla de cierre fijadas antes de mirar.

**Regla:** Un récord plantea una pregunta; una muestra representativa, un intervalo y una prueba predefinida sostienen la decisión.